In [1]:
import pandas as pd
import numpy as np
import kagglehub
import re

In [2]:
path = kagglehub.dataset_download("zusmani/pakistans-largest-ecommerce-dataset")
file_path = f"{path}/Pakistan Largest Ecommerce Dataset.csv"
df = pd.read_csv(file_path, low_memory=False)
df

100%|██████████| 13.8M/13.8M [00:00<00:00, 91.3MB/s]

Extracting files...


,item_id,status,created_at,sku,price,qty_ordered,grand_total,increment_id,category_name_1,sales_commission_code,...,Month,Customer Since,M-Y,FY,Customer ID,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,211131.0,complete,7/1/2016,kreations_YI 06-L,1950.0,1.0,1950.0,100147443,Women's Fashion,\N,...,7.0,2016-7,7-2016,FY17,1.0,NaN,NaN,NaN,NaN,NaN
1,211133.0,canceled,7/1/2016,kcc_Buy 2 Frey Air Freshener & Get 1 Kasual Bo...,240.0,1.0,240.0,100147444,Beauty & Grooming,\N,...,7.0,2016-7,7-2016,FY17,2.0,NaN,NaN,NaN,NaN,NaN
2,211134.0,canceled,7/1/2016,Ego_UP0017-999-MR0,2450.0,1.0,2450.0,100147445,Women's Fashion,\N,...,7.0,2016-7,7-2016,FY17,3.0,NaN,NaN,NaN,NaN,NaN
3,211135.0,complete,7/1/2016,kcc_krone deal,360.0,1.0,60.0,100147446,Beauty & Grooming,R-FSD-52352,...,7.0,2016-7,7-2016,FY17,4.0,NaN,NaN,NaN,NaN,NaN
4,211136.0,order_refunded,7/1/2016,BK7010400AG,555.0,2.0,1110.0,100147447,Soghaat,\N,...,7.0,2016-7,7-2016,FY17,5.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048570,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1048571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1048572,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1048573,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.shape, df.columns, df.dtypes

((1048575, 26),
 Index(['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered',
        'grand_total', 'increment_id', 'category_name_1',
        'sales_commission_code', 'discount_amount', 'payment_method',
        'Working Date', 'BI Status', ' MV ', 'Year', 'Month', 'Customer Since',
        'M-Y', 'FY', 'Customer ID', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23',
        'Unnamed: 24', 'Unnamed: 25'],
       dtype='object'),
 item_id                  float64
 status                    object
 created_at                object
 sku                       object
 price                    float64
 qty_ordered              float64
 grand_total              float64
 increment_id              object
 category_name_1           object
 sales_commission_code     object
 discount_amount          float64
 payment_method            object
 Working Date              object
 BI Status                 object
  MV                       object
 Year                     float64
 Month        

In [4]:
df.isna().sum()

,0
item_id,464051
status,464066
created_at,464051
sku,464071
price,464051
qty_ordered,464051
grand_total,464051
increment_id,464051
category_name_1,464215
sales_commission_code,601229


In [5]:
df = df.dropna(how='all')
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
df = df[df['BI Status'] != '#REF!']

In [6]:
def standardize_columns(c):
    c = str(c).strip()
    c = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', c)
    c = re.sub(r'[^a-zA-Z0-9]+', '_', c)
    return re.sub(r'_+', '_', c).strip('_').lower()
df.columns = [standardize_columns(c) for c in df.columns]

In [7]:
df.columns

Index(['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered',
       'grand_total', 'increment_id', 'category_name_1',
       'sales_commission_code', 'discount_amount', 'payment_method',
       'working_date', 'bi_status', 'mv', 'year', 'month', 'customer_since',
       'm_y', 'fy', 'customer_id'],
      dtype='object')

In [8]:
df['discount_amount'] = df['discount_amount'].fillna(0)
df['net_revenue'] = (df['price'] * df['qty_ordered']) - df['discount_amount']

In [9]:
df['net_revenue']

,net_revenue
1,240.0
2,2450.0
3,60.0
4,1110.0
5,80.0
...,...
584519,699.0
584520,35599.0
584521,259998.0
584522,174600.0


In [10]:
cols_to_drop = ['grand_total', 'm_y', 'fy', 'mv', 'working_date', 'year', 'month', 'sales_commission_code']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

In [11]:
df['status'].value_counts()


,count
status,
complete,233684
canceled,201249
received,77290
order_refunded,59529
refund,8050
cod,2859
paid,1159
closed,494
payment_review,57


In [12]:
df['payment_method'].value_counts()


,count
payment_method,
cod,271959
Payaxis,97641
Easypay,82900
jazzwallet,35145
easypay_voucher,31176
bankalfalah,23065
jazzvoucher,15633
Easypay_MA,14028
customercredit,7555


In [13]:
df['category_name_1'].value_counts()

,count
category_name_1,
Mobiles & Tablets,115710
Men's Fashion,92221
Women's Fashion,59720
Appliances,52413
Superstore,43613
Beauty & Grooming,41496
Soghaat,34011
Others,29218
Home & Living,26504


In [14]:
df['status'] = df['status'].str.lower().str.strip()
df['payment_method'] = df['payment_method'].str.lower().str.strip()

status_map = {
    'received': 'complete',
    'paid': 'complete',
    'closed': 'complete',
    'cod': 'complete',
    'order_refunded': 'refunded',
    'refund': 'refunded',
    'exchange': 'refunded'
}
df['status'] = df['status'].replace(status_map)

payment_map = {
    'easypay': 'mobile_wallet',
    'jazzwallet': 'mobile_wallet',
    'easypay_voucher': 'voucher',
    'jazzvoucher': 'voucher',
    'bankalfalah': 'bank_transfer',
    'payaxis': 'bank_transfer',
    'apg': 'bank_transfer',
    'mcblite': 'bank_transfer',
    'internetbanking': 'bank_transfer'
}
df['payment_method'] = df['payment_method'].replace(payment_map)
df['payment_method']

,payment_method
1,cod
2,cod
3,cod
4,cod
5,cod
...,...
584519,cod
584520,bank_transfer
584521,bank_transfer
584522,bank_transfer


In [15]:
q1 = df['price'].quantile(0.25)
q3 = df['price'].quantile(0.75)

iqr = q3 - q1

upper_bound = q3 + (1.5 * iqr)

df = df[df['price'] <= upper_bound]
df['price']

,price
1,240.0
2,2450.0
3,360.0
4,555.0
5,80.0
...,...
584515,999.0
584516,6760.0
584517,400.0
584518,499.0


In [16]:
df['created_at'] = pd.to_datetime(df['created_at'])

df['sales_event'] = 'Normal'

white_friday_mask = (df['created_at'] >= '2017-11-20') & (df['created_at'] <= '2017-11-26')
df.loc[white_friday_mask, 'sales_event'] = 'White Friday'

ramadan_mask = (df['created_at'] >= '2017-05-27') & (df['created_at'] <= '2017-06-24')
df.loc[ramadan_mask, 'sales_event'] = 'Ramadan'


In [17]:
filtered_df = df[df['sales_event'].isin(['Ramadan', 'White Friday'])]
print(filtered_df)

         item_id    status created_at  \
232737  499951.0  canceled 2017-05-27   
232738  499952.0  complete 2017-05-27   
232739  499953.0  complete 2017-05-27   
232741  499954.0  canceled 2017-05-27   
232742  499958.0  canceled 2017-05-27   
...          ...       ...        ...   
403824  699174.0  complete 2017-11-26   
403826  699170.0  complete 2017-11-26   
403829  699176.0  refunded 2017-11-26   
403830  699177.0  refunded 2017-11-26   
403832  699178.0  complete 2017-11-26   

                                               sku   price  qty_ordered  \
232737                           SN_Muzadati-500gm   170.0          6.0   
232738  vitamin_TVC-Dark Circles + Vitamin E Night   990.0          1.0   
232739                          BE_Travel Wash Bag   490.0          1.0   
232741                          Romoss_20-20000mAh  2444.0          1.0   
232742                           SN_Muzadati-500gm   170.0          6.0   
...                                            ...     ..

In [18]:
df.columns

Index(['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered',
       'increment_id', 'category_name_1', 'discount_amount', 'payment_method',
       'bi_status', 'customer_since', 'customer_id', 'net_revenue',
       'sales_event'],
      dtype='object')

In [19]:
df = df.dropna(subset=['customer_id']).reset_index(drop=True)

df = df[df['price'] > 0].reset_index(drop=True)

df['customer_id'] = df['customer_id'].astype(int)
df['qty_ordered'] = df['qty_ordered'].astype(int)

In [20]:
# Costumer Table
dim_customers = df[['customer_id', 'customer_since']].drop_duplicates().dropna(subset=['customer_id'])

# Product Table
dim_products = df[['sku', 'category_name_1', 'price']].drop_duplicates(subset=['sku']).dropna(subset=['sku'])

# Fact Table
fact_columns = ['item_id', 'increment_id', 'customer_id', 'sku', 'created_at', 'qty_ordered', 'net_revenue', 'status', 'payment_method', 'sales_event']
fact_orders = df[fact_columns]

dim_customers.to_csv('Dim_Customers.csv', index=False)
dim_products.to_csv('Dim_Products.csv', index=False)
fact_orders.to_csv('Fact_Orders.csv', index=False)

print(f"Fact Orders Shape: {fact_orders.shape}")

Fact Orders Shape: (476593, 10)


In [21]:
!pip install duckdb
import duckdb
con = duckdb.connect(':memory:').register('fact_orders', fact_orders).register('dim_customers', dim_customers).register('dim_products', dim_products)

In [22]:
query1="""
SELECT * FROM fact_orders """
duckdb.query(query1).df()

,item_id,increment_id,customer_id,sku,created_at,qty_ordered,net_revenue,status,payment_method,sales_event
0,211133.0,100147444,2,kcc_Buy 2 Frey Air Freshener & Get 1 Kasual Bo...,2016-07-01,1,240.0,canceled,cod,Normal
1,211134.0,100147445,3,Ego_UP0017-999-MR0,2016-07-01,1,2450.0,canceled,cod,Normal
2,211135.0,100147446,4,kcc_krone deal,2016-07-01,1,60.0,complete,cod,Normal
3,211136.0,100147447,5,BK7010400AG,2016-07-01,2,1110.0,refunded,cod,Normal
4,211137.0,100147448,6,UK_Namkino All In One 200 Gms,2016-07-01,1,80.0,canceled,cod,Normal
...,...,...,...,...,...,...,...,...,...,...
476588,905198.0,100562381,111132,MEFPAK5B360B03C6B72,2018-08-28,1,999.0,complete,customercredit,Normal
476589,905199.0,100562382,8123,MATINF59BAB39FDBEF1,2018-08-28,2,13520.0,pending,voucher,Normal
476590,905200.0,100562383,115325,WOFVAL59D5EA84167F9-M,2018-08-28,1,400.0,complete,cod,Normal
476591,905202.0,100562384,115325,WOFNIG5B4D7EB0E9FDD-L,2018-08-28,1,499.0,complete,cod,Normal


In [23]:
query2 = """
SELECT
    status,
    SUM(net_revenue) AS total_revenue
FROM fact_orders
GROUP BY status
ORDER BY total_revenue DESC;
"""
duckdb.query(query2).df()

,status,total_revenue
0,complete,3.337390e+08
1,canceled,3.321787e+08
2,refunded,8.231492e+07
3,payment_review,1.120387e+05
4,pending,8.218600e+04
5,processing,2.896770e+04
6,holded,2.501950e+04
7,None,2.242900e+04
8,pending_paypal,7.565500e+03
9,\n,1.620000e+02


In [24]:
query3 = """
SELECT
    payment_method,
    COUNT(item_id) AS total_orders,
    SUM(CASE WHEN status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
    ROUND(
        100.0 * SUM(CASE WHEN status = 'canceled' THEN 1 ELSE 0 END) / NULLIF(COUNT(item_id), 0),
        2
    ) AS cancellation_rate_pct
FROM fact_orders
GROUP BY payment_method
ORDER BY cancellation_rate_pct DESC;
"""
duckdb.query(query3).df()

,payment_method,total_orders,canceled_orders,cancellation_rate_pct
0,mygateway,504,488.0,96.83
1,ublcreditcard,532,359.0,67.48
2,easypay_ma,11117,7212.0,64.87
3,bank_transfer,88008,53509.0,60.80
4,mobile_wallet,93554,52878.0,56.52
5,voucher,22524,11483.0,50.98
6,productcredit,121,10.0,8.26
7,cod,252603,16375.0,6.48
8,cashatdoorstep,711,5.0,0.70
9,customercredit,6867,45.0,0.66


In [25]:
query_3 = """
SELECT
    fo.sales_event,
    COUNT(fo.item_id) AS total_items_sold,
    SUM(fo.net_revenue) AS total_revenue
FROM fact_orders fo
JOIN dim_products dp ON fo.sku = dp.sku
WHERE fo.sales_event IN ('White Friday', 'Normal')
GROUP BY fo.sales_event
ORDER BY total_revenue DESC;
"""
duckdb.query(query_3).df()

,sales_event,total_items_sold,total_revenue
0,Normal,428772,6.299313e+08
1,White Friday,29287,4.493556e+07


In [26]:
query_4 = """
SELECT
    dp.category_name_1 AS category,
    SUM(fo.net_revenue) AS total_revenue
FROM fact_orders fo
JOIN dim_products dp ON fo.sku = dp.sku
WHERE fo.sales_event = 'Ramadan'
GROUP BY dp.category_name_1
ORDER BY total_revenue DESC
LIMIT 5;
"""
duckdb.query(query_4).df()

,category,total_revenue
0,Mobiles & Tablets,5.644401e+07
1,Women's Fashion,3.659382e+06
2,Men's Fashion,3.516576e+06
3,Appliances,2.239089e+06
4,Beauty & Grooming,2.065474e+06


In [27]:
query_5 = """
WITH DailyMetrics AS (
    SELECT
        CAST(created_at AS DATE) AS sales_date,
        COUNT(DISTINCT increment_id) AS total_orders,
        COUNT(DISTINCT customer_id) AS unique_buyers,
        SUM(net_revenue) AS daily_revenue
    FROM fact_orders
    WHERE status = 'complete'
    GROUP BY CAST(created_at AS DATE)
)
SELECT
    sales_date,
    total_orders,
    unique_buyers,
    ROUND(daily_revenue, 2) AS daily_revenue,
    ROUND(daily_revenue / NULLIF(total_orders, 0), 2) AS calculated_aov
FROM DailyMetrics
ORDER BY sales_date DESC
LIMIT 10;
"""
duckdb.query(query_5).df()

,sales_date,total_orders,unique_buyers,daily_revenue,calculated_aov
0,2018-08-28,22,21,38975.75,1771.63
1,2018-08-27,88,84,180947.92,2056.23
2,2018-08-26,57,57,131268.40,2302.95
3,2018-08-25,75,70,141796.80,1890.62
4,2018-08-24,69,59,197617.10,2864.02
5,2018-08-23,42,40,77732.10,1850.76
6,2018-08-22,21,21,33215.00,1581.67
7,2018-08-21,49,49,99830.60,2037.36
8,2018-08-20,60,55,190172.40,3169.54
9,2018-08-19,73,65,157944.50,2163.62


In [31]:
query_6 = """
WITH MonthlyRevenue AS (
    SELECT
        DATE_TRUNC('month', created_at) AS sales_month,
        SUM(net_revenue) AS current_month_revenue
    FROM fact_orders
    WHERE status = 'complete'
    GROUP BY DATE_TRUNC('month', created_at)
)
SELECT
    sales_month,
    ROUND(current_month_revenue, 2) AS current_revenue,
    ROUND(
        LAG(current_month_revenue, 1) OVER (ORDER BY sales_month ASC),
        2
    ) AS prior_month_revenue,
    ROUND(
        current_month_revenue - LAG(current_month_revenue, 1) OVER (ORDER BY sales_month ASC),
        2
    ) AS mom_absolute_growth,
    ROUND(
        100.0 * (current_month_revenue - LAG(current_month_revenue, 1) OVER (ORDER BY sales_month ASC))
        / NULLIF(LAG(current_month_revenue, 1) OVER (ORDER BY sales_month ASC), 0),
        2
    ) AS mom_growth_pct
FROM MonthlyRevenue
ORDER BY sales_month ASC;
"""
duckdb.query(query_6).df()

,sales_month,current_revenue,prior_month_revenue,mom_absolute_growth,mom_growth_pct
0,2016-07-01,4464306.09,NaN,NaN,NaN
1,2016-08-01,4971587.87,4464306.09,507281.78,11.36
2,2016-09-01,7064903.01,4971587.87,2093315.14,42.11
3,2016-10-01,5312343.97,7064903.01,-1752559.04,-24.81
4,2016-11-01,30043620.66,5312343.97,24731276.69,465.54
5,2016-12-01,6649924.38,30043620.66,-23393696.28,-77.87
6,2017-01-01,8867606.17,6649924.38,2217681.79,33.35
7,2017-02-01,10925819.88,8867606.17,2058213.71,23.21
8,2017-03-01,16734528.82,10925819.88,5808708.94,53.16
9,2017-04-01,10902359.75,16734528.82,-5832169.07,-34.85


In [29]:
query_7 = """
WITH CategoryRevenue AS (
    SELECT
        dp.category_name_1 AS category,
        SUM(fo.net_revenue) AS total_revenue,
        RANK() OVER (ORDER BY SUM(fo.net_revenue) DESC) AS revenue_rank
    FROM fact_orders fo
    JOIN dim_products dp ON fo.sku = dp.sku
    WHERE fo.status = 'complete'
    GROUP BY dp.category_name_1
)
SELECT
    category,
    ROUND(total_revenue, 2) AS total_revenue,
    revenue_rank
FROM CategoryRevenue
WHERE revenue_rank <= 3
ORDER BY revenue_rank ASC;
"""
duckdb.query(query_7).df()

,category,total_revenue,revenue_rank
0,Mobiles & Tablets,70161727.47,1
1,Appliances,51879976.72,2
2,Women's Fashion,49104961.03,3


In [30]:
query_8 = """
SELECT DISTINCT
    customer_id,
    MIN(created_at) OVER (PARTITION BY customer_id) AS first_order_date
FROM fact_orders
ORDER BY customer_id ASC
LIMIT 10;
"""
duckdb.query(query_8).df()

,customer_id,first_order_date
0,2,2016-07-01
1,3,2016-07-01
2,4,2016-07-01
3,5,2016-07-01
4,6,2016-07-01
5,7,2016-07-01
6,9,2016-07-01
7,10,2016-07-01
8,11,2016-07-01
9,12,2016-07-01
